# Learned baselines on Colab — one method per session, meshes saved to Drive

Runs a learned reconstruction baseline (Neural-Pull / CAP-UDF / ConvONet) on the **same 7 clouds** our model is
compared on (shipped in the repo), and writes the output meshes to your Google Drive so the main notebook
(`colab_train.ipynb`) can score them into the comparison.

**Why one method per session:** each method pins a different, *older* torch/CUDA (1.8–1.13) that has no wheel for
Colab's Python 3.11. We use **condacolab** to build the method's exact Python+torch env. Switching method ⇒
**Runtime → Restart session**, change `METHOD`, run again.

**Honest scope:** Neural-Pull and CAP-UDF are *per-shape optimizers* (no weights, no domain gap — the fair
apples-to-apples). ConvONet is *feed-forward with ShapeNet weights* — **zero-shot/OOD** on our shapes. POCO and
NKSR need heavy custom builds (POCO: conda Python 3.7; NKSR: openvdb/CUDA package build) — run those via their
Docker images on a Docker host (`baselines_ext/<m>/`); they are not wired into this notebook.

## 1· Install condacolab  (this RESTARTS the kernel — run the cells below *after* it restarts)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # kernel restarts automatically here

## 2· Configure (run everything from here after the restart)

In [ ]:
METHOD   = "neuralpull"     # neuralpull | capudf | convonet
DRIVE_WS = "/content/drive/MyDrive/wsn_train"
REPO_URL = "https://github.com/OlegJakushkin/Points_as_supertoroids.git"
BRANCH   = "main"
MAXITER  = 40000           # per-shape optimisers (neuralpull): iterations per shape (lower = faster, rougher)
print("method:", METHOD)

## 3· Mount Drive, clone the repo (clouds + adapters), define the per-method env

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir("/content/wsn"):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "/content/wsn"], check=True)

SRC = "/content/m_repo"    # where the method's official repo is cloned
SPEC = {
    "neuralpull": dict(py="3.8", fair=True, conda=[], env={"NEURALPULL_DIR": SRC}, run_args=f"--maxiter {MAXITER}",
        setup=[
            "pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113",
            "pip install numpy==1.22.4 scipy==1.8.1 tqdm pyhocon==0.3.57 trimesh PyMCubes",
            f"git clone --depth 1 https://github.com/mabaorui/NeuralPull-Pytorch {SRC}"]),
    "capudf": dict(py="3.8", fair=True, conda=["cudatoolkit-dev=11.3 -c conda-forge"], env={"PYTHONPATH": SRC}, run_args="",
        setup=[
            "pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113",
            "pip install numpy==1.23.5 cython tqdm pyhocon==0.3.57 trimesh PyMCubes scipy point_cloud_utils==0.29.7",
            f"git clone https://github.com/junshengzhou/CAP-UDF {SRC} && cd {SRC} && git checkout 0b360a3",
            f"cd {SRC}/extensions/chamfer_dist && python setup.py install",
            f"sed -i '/from tkinter import Variable/d; /from random import sample/d' {SRC}/tools/utils.py"]),
    "convonet": dict(py="3.8", fair=False, conda=[], env={"PYTHONPATH": SRC, "TORCH_HOME": "/root/.cache/torch"}, run_args="",
        setup=[
            "pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117",
            "pip install torch-scatter==2.1.1 -f https://data.pyg.org/whl/torch-1.13.1+cu117.html",
            "pip install Cython==0.29.36 numpy==1.23.5 scipy==1.9.3 scikit-image==0.19.3 trimesh==3.22.4 plyfile==0.7.4 PyYAML==6.0 tqdm==4.65.0 pandas==1.4.4 tensorboardX==2.6",
            f"git clone https://github.com/autonomousvision/convolutional_occupancy_networks {SRC} && cd {SRC} && git checkout 838bea5b2f1314f2edbb68d05ebb0db49f1f3bd2 && python setup.py build_ext --inplace"]),
}
assert METHOD in SPEC, f"{METHOD} is not wired into this notebook (POCO/NKSR: use their Docker images). Pick {list(SPEC)}"
spec = SPEC[METHOD]
print(METHOD, "| fair apples-to-apples" if spec["fair"] else "| zero-shot/OOD (ShapeNet weights)")

## 4· Build the method's conda env, run it on the 7 clouds, save meshes to Drive

In [ ]:
import os, glob, shlex, shutil, subprocess
ENV = f"bl_{METHOD}"

def sh(cmd):
    print("==>", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

sh(f"conda create -y -n {ENV} python={spec['py']}")
for c in spec["conda"]:
    sh(f"conda install -y -n {ENV} {c}")
for cmd in spec["setup"]:
    sh(f"conda run -n {ENV} bash -lc {shlex.quote(cmd)}")

CLOUDS = "/content/wsn/baselines_ext/clouds"
OUT = f"/content/out_{METHOD}"; os.makedirs(OUT, exist_ok=True)
envstr = " ".join(f'{k}="{v}"' for k, v in spec["env"].items())
inner = f"{envstr} python /content/wsn/baselines_ext/{METHOD}/run.py {CLOUDS} {OUT} {spec['run_args']}"
sh(f"conda run -n {ENV} bash -lc {shlex.quote(inner)}")

dst = f"{DRIVE_WS}/results/learned/{METHOD}"; os.makedirs(dst, exist_ok=True)
n = 0
for f in glob.glob(f"{OUT}/*.ply"):
    shutil.copy2(f, dst); n += 1
print(f"\nsaved {n} meshes -> {dst}")

## 5· Next
Restart the session and set `METHOD` to the next one (`capudf`, `convonet`). When you have the meshes for the
methods you want, **score them in `colab_train.ipynb`** (its learned-baselines cell points `BL_OUT` at
`DRIVE_WS/results/learned` and runs `compare/fig_learned.py` to render `cmp_learned.png` + the metrics table
against our model).